## Setting up Gmail's API from GCP
Go to Google Cloud Console, and in that go to Marketplace and enable the API. 
Then in "API & Services", go to credentials and under "OAuth 2.0 Client IDs", create your new credentials.
Once created, download the `credentials.json` file.
Then go to "Google Auth Platform" > "Audience", and there add your email in the "Test users".

Now add the credentials.json in your project, as you're calling that file, and also add it in .gitignore.

In [2]:
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from email.mime.text import MIMEText
import pickle
import os
import base64
from bs4 import BeautifulSoup
from openai import OpenAI
from agents import Agent, Runner, trace, function_tool, handoff, HandoffInputData, RunHooks, ModelSettings
import json
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pypdf import PdfReader
import re

In [3]:
load_dotenv(override=True)

True

In [4]:
openai_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI()

In [5]:
# This will open the browser once and do the OAuth.

SCOPES = ['https://www.googleapis.com/auth/gmail.modify']

def get_gmail_service():
    creds = None

    if os.path.exists('token.pkl'):
        with open('token.pkl', 'rb') as token:
            creds = pickle.load(token)

    if not creds or not creds.valid:
        flow = InstalledAppFlow.from_client_secrets_file(
            '/Users/manuyadav/projects/email_agent/credentials.json', SCOPES)
        creds = flow.run_local_server(port=0)

        with open('token.pkl', 'wb') as token:
            pickle.dump(creds, token)

    return build('gmail', 'v1', credentials=creds)

In [6]:
# This is to read emails

def extract_email_data(service, msg_id):
    msg = service.users().messages().get(
        userId='me',
        id=msg_id,
        format='full'
    ).execute()

    payload = msg['payload']
    headers = payload.get('headers', [])

    # -----------------------
    # ID and Subject
    # -----------------------
    msg_id = msg['id']
    subject = next(
        (h['value'] for h in headers if h['name'] == 'Subject'),
        ''
    )

    # -----------------------
    # Snippet (easy win)
    # -----------------------
    snippet = msg.get('snippet', '')

    # -----------------------
    # Extract full body
    # -----------------------
    def get_body(payload):
        mime = payload.get("mimeType", "")
        body = payload.get("body", {})

        if mime == "text/plain" and "data" in body:
            return body["data"], "plain"

        if mime == "text/html" and "data" in body:
            return body["data"], "html"

        for part in payload.get("parts", []):
            result = get_body(part)
            if result:
                return result

        return None

    result = get_body(payload)

    full_text = ""

    if result:
        data, body_type = result

        decoded = base64.urlsafe_b64decode(data).decode(
            "utf-8", errors="ignore"
        )

        if body_type == "html":
            soup = BeautifulSoup(decoded, "html.parser")
            full_text = soup.get_text(separator="\n")
        else:
            full_text = decoded

    return {
		  "id": msg_id,
        "subject": subject,
        "snippet": snippet,
        "body": full_text.strip()
    }

In [7]:
# This is to send emails

def send_email(service, to, subject, body):
    message = MIMEText(body)

    message['to'] = to
    message['subject'] = subject

    raw = base64.urlsafe_b64encode(message.as_bytes()).decode()

    message = {'raw': raw}

    service.users().messages().send(
        userId='me',
        body=message
    ).execute()

    print("Email sent!")

In [8]:
# This to write drafts (this will be called by function tool)

def create_reply_draft(service, to, subject, body, thread_id, message_id):
	try:
		message = MIMEText(body)

		message['to'] = to
		message['subject'] = f"Re: {subject}"
		message['In-Reply-To'] = message_id
		message['References'] = message_id

		raw = base64.urlsafe_b64encode(
			message.as_bytes()
		).decode()

		draft = {
			'message': {
				'raw': raw,
				'threadId': thread_id
			}
		}

		service.users().drafts().create(
			userId='me',
			body=draft
		).execute()

		return {'is_draft_written': True}
	
	except Exception as e:
		return {'is_draft_written': False,
		'Error': str(e)}

In [9]:
service = get_gmail_service()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=590520607160-6odmtjdd2nnv7b6cccj4aetq1hfabulb.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A59708%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.modify&state=cGwc3Y5OIe8q2w73aansuNNstd98Hc&code_challenge=rlXWb06nImxdmvnRCV2WuZJvrdJCfyMpqVMbViRGScU&code_challenge_method=S256&access_type=offline


### Util Functions

In [10]:
def payload_builder(output_type):
	def _on_handoff(ctx, filtered: output_type):
		"""Required by SDK when input_type is set; no shared state needed."""
		return None


	def _input_filter(data: HandoffInputData) -> HandoffInputData:
		"""Pass only classifier handoff payload to agent 3 (drop full history)."""
		handoff_call = next(
			(item for item in reversed(data.new_items) if getattr(item, "type", "") == "handoff_call_item"),
			None,
		)

		if handoff_call is None:
			return data.clone(input_history=(), pre_handoff_items=(), new_items=())

		payload_json = handoff_call.raw_item.arguments
		return data.clone(
			input_history=({"role": "user", "content": payload_json},),
			pre_handoff_items=(),
			new_items=(),
		)
	
	return _on_handoff, _input_filter

In [11]:
class PrintAgentOutputs(RunHooks):
	async def on_agent_start(self, context, agent):
		print(f"\nAgent {agent.name} is running ...")
		
	async def on_llm_end(self, context, agent, response):
		for item in response.output:
			if item.type == 'function_call':
				print(f"Tool invoked: {item.name}")
				print(f"Arguments: {item.arguments}")

				data_dict = json.loads(item.arguments)
				
				if 'emails' in data_dict:
					print(f"Total {len(data_dict['emails'])} emails")
			else:
				print(f"\nFinal output: {item.content[0].text}")
			 
	async def on_agent_end(self, context, agent, output):
		print(f"Task Finished")

### Fifth Agent (writes the draft in Gmail)

In [12]:
# This is used for extracting the info of the email, which will be used for drafting the email reply.

class Drafts(BaseModel):
	id: str = Field(description='Message id of each email.')
	draft: str = Field(description='Email draft which LLM will create.')

@function_tool
def extract_info_and_write_draft(drafts: list[Drafts]):

	for i in drafts:
		msg = service.users().messages().get(
			userId='me',
			id=i.id,
			format='full'
		).execute()

		thread_id = msg['threadId']

		# Extract Message-ID header (important for reply)
		headers = msg['payload']['headers']
		message_id_header = next(
			(h['value'] for h in headers if h['name'] == 'Message-ID'),
			None
		)

		# Extract sender email (you need this for reply)
		from_header = next(
			(h['value'] for h in headers if h['name'] == 'From'),
			None
		)

		if not message_id_header or not from_header:
			return (f"Missing required headers, skipping email with id: {i.id}")

		# Cleaning the sender email by regex
		to_email = re.search(r'<(.+?)>', from_header)
		to_email = to_email.group(1) if to_email else from_header

		# 
		response = create_reply_draft(
			service=service,
			to=to_email,
			subject="Interview Opportunity",
			body=i.draft,
			thread_id=thread_id,
			message_id=message_id_header
		)
	
	return response


# For testing
# extract_info_and_write_draft('19e894a5acb0edb6', 'Yo nigga')

In [13]:
write_draft_instruction = """You are an execution agent.

Your only responsibility is to create a draft reply email using the provided tool.

You will receive:
- msg_id: the ID of the email to reply to
- draft: the reply content to be written

Instructions:

1. Call the tool `extract_info_and_write_draft`
2. Pass the FULL list of draft exactly as received (do not modify, filter, or recreate)
3. Do NOT modify, analyze, or rewrite the draft content.
4. Simply return if draft is written in email or not.

Important rules:
- Do NOT generate or change the draft text
- Do NOT skip the tool call
- Do NOT add explanations or extra text
- Your job is only to execute the tool call with the given inputs"""

In [14]:
write_draft_agent = Agent(
	name='write_draft_agent',
	model='gpt-4o-mini',
	instructions=write_draft_instruction,
	tools=[extract_info_and_write_draft],
	handoff_description='Execute the tool'
)

### Fourth Agent (Writing the draft email)

In [15]:
with open("/Users/manuyadav/projects/email_agent/me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()
	 

reader = PdfReader("/Users/manuyadav/projects/email_agent/me/manu_yadav_cv.pdf")
resume = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text

name = "Manu Yadav"

In [16]:
draft_generator_instruction = f"""
You are acting as {name}, a job candidate communicating with recruiters over email.

You will receive a list of emails. Each email contains:
- id: unique identifier
- body: full email content

Your task is to generate a professional reply for emails. And do not forget to call the handoff in the end.

---

### Candidate Summary:
{summary}

---

### Resume:
{resume}

---

### Emails to Respond:
Input which came from previous tool i.e. fetch_full_emails.

---

### Instructions:

- For each email:
  - Understand the content
  - Draft a professional reply
  - Use only the provided resume and summary as source of truth
  - Do NOT hallucinate missing details

- Maintain a professional, confident tone
- Be concise but informative
- If an email is irrelevant or not job-related, skip it
- Include only emails that require a reply

### Handoff instructions:

- Do not skip the handoff and transfer the results to next agent i.e. write_draft_agent
- Do not call multiple handoffs, but only once when you draft the reply of all emails.
- You MUST have to call the handoff.

"""

In [17]:
class Drafts(BaseModel):
	id: str = Field(description='Message id of each email.')
	draft: str = Field(description='Email draft which LLM will create.')

class DraftList(BaseModel):
	emails: list[Drafts]

In [18]:
draft_generator_on_handoff, draft_generator_input_filter = payload_builder(DraftList)


draft_generator_agent = Agent(
    name="draft_generator_agent",
    model="gpt-4o-mini",
    model_settings=ModelSettings(parallel_tool_calls=False),
    instructions=draft_generator_instruction,
    handoffs=[
        handoff(
            write_draft_agent,
            input_type=DraftList,
            on_handoff=draft_generator_on_handoff,
            input_filter=draft_generator_input_filter,
            tool_description_override=(
                "Call this exactly once after all drafts are generated. "
                "Pass one JSON object with emails as a list containing every draft."
            ),
        )
    ],
)

### Third Agent (New)
This is only for pulling the full emails which are relevent

In [19]:
class EmailResult(BaseModel):
	msg_id: str
	subject: str
	feedback: str
	
@function_tool
def fetch_full_emails(emails: list[EmailResult]) -> list[dict]:

    print(f"{len(emails)} emails are relevent.")
    print([{i.subject} for i in emails])
    print("Fetching full emails ...")
    
    msg_ids = [e.msg_id for e in emails]
	 
    query = "is: newer_than:2d -category:promotions -category:social"

    results = service.users().messages().list(
        userId='me',
        q=query,
        maxResults=10
    ).execute().get('messages', [])

    emails = []

    for m in results:
      if m['id'] in msg_ids:
        data = extract_email_data(service, m['id'])
        emails.append({
            "id": m["id"],
            "subject": data["subject"],
				"body": data["body"]
        })
	
    print("Full emails fetched")

    return emails

In [20]:
full_email_fetch_prompt = '''You are responsible for fetching full email content based on filtered email inputs.

You will receive:
A list of relevant emails from the previous agent, each containing:
- msg_id
- subject
- feedback

## Your Task
1. Call the tool `fetch_full_emails`
2. Pass the FULL list of emails exactly as received (do not modify, filter, or recreate)
3. The tool will return full email data (id, subject, body)
4. Handoff the results in the required fomrat to next agent i.e. email_fetcher_and_drafter_agent
5. Even if email body are empty, even then pass them to next handoff.

## Important Rules
- ALWAYS call the tool if input list is non-empty
- NEVER fabricate or simulate email content
- NEVER skip the tool call
- NEVER change message IDs
- Dont skip the handoff

## After Tool Execution
Return the result in the required structured format to next agent via handoff

## Edge Case
- If input list is empty → return:
{
  "emails": []
}
and DO NOT call the tool
'''

In [21]:
class FullEmails(BaseModel):
	id: str = Field(description='Message id of each email.')
	subject: str = Field(description='Subject of the email')
	body: str = Field(description='Full email')

class FullEmailsList(BaseModel):
	emails: list[FullEmails]


full_email_on_handoff, full_email_input_filter = payload_builder(FullEmailsList)

full_email_fetch_agent = Agent(
   name='full_email_fetch_agent',
	instructions=full_email_fetch_prompt,
	model='gpt-4o-mini',
	tools=[fetch_full_emails],
	handoffs=[
        handoff(
            draft_generator_agent,
            input_type=FullEmailsList,
            on_handoff=full_email_on_handoff,
            input_filter=full_email_input_filter,
            tool_description_override=(
					 "Hand off full email to next agent"
					 "Pass id, subject, and body for each"
            ),
        )
    ],
	handoff_description='Fetch the full emails',
	# output_type=FullEmailsList
)

### Second Agent (Agent for filtering the emails)
This is responsible for filtering which emails are from recruiters, which needs my attention.

In [22]:
classification_instruction = f"""You are an email classification assistant.

You will receive a list of emails in structured format. Each email contains:
- msg_id: unique identifier
- subject: subject line
- snippet: short preview of the email

## Your Task
Identify ONLY the emails that are relevant to job opportunities or professional communication.

## Criteria for Relevance
Mark an email as relevant ONLY if it clearly includes:
- Messages from recruiters, hiring managers, or HR
- Job opportunities, interview invitations, or follow-ups
- Requests related to job applications (documents, scheduling, etc.)

## Ignore
Do NOT include emails such as:
- Promotions, newsletters, marketing content
- Social media notifications
- Automated updates
- Any non-job-related communication

## Important Rules (STRICT)
- Be highly selective (precision > recall)
- If unsure, EXCLUDE the email
- NEVER include all emails unless ALL are clearly relevant
- It is completely valid to return an empty list

## Handoff Rules (CRITICAL — do not return JSON as your final message)
- If ZERO emails are relevant → reply with plain text: "No relevant emails" and do NOT call any handoff tool
- If one or more emails are relevant → you MUST call the tool `transfer_to_full_email_fetch_agent` with a payload containing ONLY those emails
- Do NOT return classification JSON as your final response — the handoff tool call is how you pass results forward

Each handoff payload email must have:
- msg_id: the Gmail message id
- subject: subject line
- feedback: short reason it is relevant
"""

In [23]:
class EmailResult(BaseModel):
	msg_id: str
	subject: str
	feedback: str

class ClassifierOutput(BaseModel):
	emails: list[EmailResult]

In [24]:
on_classifier_handoff, classifier_handoff_filter = payload_builder(ClassifierOutput)

classifier_agent = Agent(
    name="classifier_agent",
    instructions=classification_instruction,
    model="gpt-4o-mini",
    handoffs=[
        handoff(
            full_email_fetch_agent,
            input_type=ClassifierOutput,
            on_handoff=on_classifier_handoff,
            input_filter=classifier_handoff_filter,
            tool_description_override=(
                "Hand off ONLY recruiter/job emails you classified as relevant. "
                "Pass msg_id, subject, and feedback for each."
            ),
        )
    ],
)

### First Agent (Main orchestrator agent)
This agent will run the function tool to fetch the latest emails and then handoff the result to second agent for classification.

In [25]:
@function_tool
def fetch_latest_emails() -> list[dict]:
    print("Fetching latest emails")
    query = "is: newer_than:2d -category:promotions -category:social -category:updates"

    results = service.users().messages().list(
        userId='me',
        q=query,
        maxResults=10
    ).execute().get('messages', [])

    emails = []

    for m in results:
        data = extract_email_data(service, m['id'])

        emails.append({
            "msg_id": m["id"],
            "subject": data["subject"],
            "snippet": data["snippet"],
        })

    print(f"{len(emails)} emails fetched.")
    print([f"Subject: {i['subject']}" for i in emails])

    return emails

In [26]:
email_fetch_instruction = """
You are a retrieval agent.

Your only responsibility is to fetch the latest emails using the available tool. 
After fetching the emails, you MUST hand off the results to the classifier_agent for further processing. Do not return the emails directly.

You have access to:
- fetch_latest_emails: Fetches recent emails from Gmail (msg_id, subject, snippet)

Instructions:

1. ALWAYS call the fetch_latest_emails tool immediately.
2. Do NOT classify or interpret the emails.
3. Simply return the tool output exactly as received.
4. Handoff the results to next agent i.e. classifier_agent

Important rules:
- Do NOT make up any data
- Do NOT skip the tool call
- Do NOT add explanations or extra text
- Your job is only to retrieve and pass the data to the next agent
"""

In [27]:
email_fetch_agent = Agent(
	name="email_fetch_agent", 
	instructions=email_fetch_instruction, 
	model="gpt-4o-mini",
	tools=[fetch_latest_emails],
	handoffs=[classifier_agent]
	)

with trace("Fetching of latest emails"):
    result = await Runner.run(email_fetch_agent, "Fetch the latest emails", hooks=PrintAgentOutputs())

	 


Agent email_fetch_agent is running ...
Tool invoked: fetch_latest_emails
Arguments: {}
Fetching latest emails
4 emails fetched.
['Subject: ', 'Subject: ', 'Subject: Fwd: Job Opportunity at Barclays | BA4 - Data Scientist', 'Subject: Fwd: Job Opportunity at KPMG |  Con\\AM - Generative AI Engineer']
Tool invoked: transfer_to_classifier_agent
Arguments: {}

Agent classifier_agent is running ...

Final output: No relevant emails.
Task Finished


In [39]:
from rich.syntax import Syntax
from rich.console import Console

code = "print('Hello world')"
syntax = Syntax(code, "python")

Console().print(syntax)

print('Hello world')                                                                                               

In [30]:
from rich import print

print(result)

RunResult(
    input='Fetch the latest emails',
    new_items=[
        ToolCallItem(
            agent=Agent(
                name='email_fetch_agent',
                handoff_description=None,
                tools=[
                    FunctionTool(
                        name='fetch_latest_emails',
                        description='',
                        params_json_schema={
                            'properties': {},
                            'title': 'fetch_latest_emails_args',
                            'type': 'object',
                            'additionalProperties': False,
                            'required': []
                        },
                        on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x115ce4b60>,
                        strict_json_schema=True,
                        is_enabled=True,
                        tool_input_guardrails=None,
                        tool_output_guardrails=None,
                        needs_approval=False,
                        timeout_seconds=None,
                        timeout_behavior='error_as_result',
                        timeout_error_function=None,
                        defer_loading=False
                    )
                ],
                mcp_servers=[],
                mcp_config={},
                instructions='\nYou are a retrieval agent.\n\nYour only responsibility is to fetch the latest 
emails using the available tool. \nAfter fetching the emails, you MUST hand off the results to the classifier_agent
for further processing. Do not return the emails directly.\n\nYou have access to:\n- fetch_latest_emails: Fetches 
recent emails from Gmail (msg_id, subject, snippet)\n\nInstructions:\n\n1. ALWAYS call the fetch_latest_emails tool
immediately.\n2. Do NOT classify or interpret the emails.\n3. Simply return the tool output exactly as 
received.\n4. Handoff the results to next agent i.e. classifier_agent\n\nImportant rules:\n- Do NOT make up any 
data\n- Do NOT skip the tool call\n- Do NOT add explanations or extra text\n- Your job is only to retrieve and pass
the data to the next agent\n',
                prompt=None,
                handoffs=[
                    Agent(
                        name='classifier_agent',
                        handoff_description=None,
                        tools=[],
                        mcp_servers=[],
                        mcp_config={},
                        instructions='You are an email classification assistant.\n\nYou will receive a list of 
emails in structured format. Each email contains:\n- msg_id: unique identifier\n- subject: subject line\n- snippet:
short preview of the email\n\n## Your Task\nIdentify ONLY the emails that are relevant to job opportunities or 
professional communication.\n\n## Criteria for Relevance\nMark an email as relevant ONLY if it clearly includes:\n-
Messages from recruiters, hiring managers, or HR\n- Job opportunities, interview invitations, or follow-ups\n- 
Requests related to job applications (documents, scheduling, etc.)\n\n## Ignore\nDo NOT include emails such as:\n- 
Promotions, newsletters, marketing content\n- Social media notifications\n- Automated updates\n- Any 
non-job-related communication\n\n## Important Rules (STRICT)\n- Be highly selective (precision > recall)\n- If 
unsure, EXCLUDE the email\n- NEVER include all emails unless ALL are clearly relevant\n- It is completely valid to 
return an empty list\n\n## Handoff Rules (CRITICAL — do not return JSON as your final message)\n- If ZERO emails 
are relevant → reply with plain text: "No relevant emails" and do NOT call any handoff tool\n- If one or more 
emails are relevant → you MUST call the tool `transfer_to_full_email_fetch_agent` with a payload containing ONLY 
those emails\n- Do NOT return classification JSON as your final response — the handoff tool call is how you pass 
results forward\n\nEach handoff payload email must have:\n- msg_id: the Gmail me